# Plotly

A self-contained refresher on **Plotly** — the declarative, JSON-backed library for
*interactive* charts (zoom, pan, hover, toggle) that render in the browser from Python.

**Domain:** Data Analysis & Research  ·  **recommended addition**  ·  **runnable:** yes

## 1. What & Why

**Plotly** turns a Python description of a chart into an interactive figure rendered by
`plotly.js` in the browser. Every figure is, underneath, a JSON spec: a list of **traces**
(the data + how to draw it) plus a **layout** (axes, title, legend, annotations). You build that
spec in Python; the JS library handles zoom, pan, hover tooltips, legend toggling, and box/lasso
selection for free.

**The problem it solves.** Matplotlib/seaborn produce static raster or vector images — great for
papers, useless for *exploring*. When you want to hover a point to read its label, zoom into a
dense region, toggle a series off, or hand a stakeholder a self-contained HTML file they can poke
at without Python installed, that's Plotly's job.

**Reach for it when** you need interactivity, a dashboard (Plotly powers **Dash**), a chart
embedded in a web page, or quick rich exploration of a DataFrame. **Skip it when** you need a
print-ready static figure (Matplotlib is simpler and lighter) or you're rendering thousands of
charts in a batch where an interactive HTML payload per chart is wasteful.

## 2. Mental Model

**A Plotly figure is a JSON document: `{data: [trace, trace, ...], layout: {...}}`.**

Everything you do is just building or mutating that dict, then handing it to `plotly.js` to draw.

```
  px.scatter(df, x=..., y=..., color=...)   ← Plotly Express: one call → whole figure
            │  (a thin factory)
            ▼
  go.Figure(data=[go.Scatter(...), ...],    ← Graph Objects: the explicit object model
            layout=go.Layout(...))             you build trace-by-trace
            │
            ▼
  fig.to_dict()  →  {"data":[...], "layout":{...}}   ← the truth; both paths produce this
            │
            ▼
  plotly.js renders it in the browser (fig.show() / fig.write_html())
```

Two ways to author the same JSON: **Plotly Express** (`px`) is the high-level "describe a tidy
DataFrame" front door — one function returns a fully-styled figure. **Graph Objects** (`go`) is
the low-level object model you reach for when you need control Express doesn't expose. They are
not rivals: `px` *returns a `go.Figure`*, so you start with `px` and keep customizing with
`fig.update_*` / `fig.add_trace`.

## 3. Key Concepts

- **Figure** — the top-level container, a `go.Figure`. It has exactly two important attributes:
  `fig.data` (a tuple of traces) and `fig.layout`.
- **Trace** — one dataset + its visual encoding: `go.Scatter`, `go.Bar`, `go.Heatmap`,
  `go.Box`, etc. A figure holds many traces, possibly of different types.
- **Layout** — everything that isn't data: title, `xaxis`/`yaxis`, legend, margins, annotations,
  shapes, color axes.
- **Plotly Express (`px`)** — high-level API: `px.scatter`, `px.line`, `px.bar`, `px.histogram`,
  `px.box`, `px.imshow`, plus faceting (`facet_col`/`facet_row`) and animation (`animation_frame`).
  Consumes a **tidy / long-form** DataFrame and column *names*.
- **Graph Objects (`go`)** — explicit traces + layout for full control and composition.
- **`update_layout` / `update_traces` / `update_xaxes`** — chainable mutators; the idiomatic way
  to tweak a figure after creation (works on both `px` and `go` figures).
- **Renderers (`plotly.io`)** — how a figure is shown: `notebook`/`plotly_mimetype` in Jupyter,
  `browser`, `png`/`svg` (needs **kaleido**), or `json`. Set via `pio.renderers.default`.
- **Export** — `fig.write_html()` (standalone interactive file), `fig.write_image()` (static, via
  kaleido), `fig.to_json()` / `fig.to_dict()` (the raw spec).

## 4. Setup

`pip install plotly` is enough for figure construction and HTML export. Static image export
(`write_image` to PNG/SVG/PDF) additionally needs **kaleido** (`pip install kaleido`). For
interactive rendering inside JupyterLab you also want a recent JupyterLab (≥ 3), which ships the
Plotly mime renderer — no separate extension needed on modern versions.

In [1]:
%pip install plotly  # add 'kaleido' for static PNG/SVG export; 'pandas' for the px examples
# Already present in this environment — the line is here for a fresh kernel.

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Make this notebook render deterministically when *executed* headlessly.
# In a live JupyterLab session you'd leave the default ("notebook"/"plotly_mimetype")
# so fig.show() / a bare `fig` produces the interactive widget.
pio.templates.default = "plotly_white"

print("plotly", plotly.__version__)
print("pandas", pd.__version__)

plotly 6.8.0
pandas 2.3.3


### A tiny, offline dataset

Plotly Express ships demo data via `px.data.*` (e.g. `px.data.gapminder()`), but some of those
download on first use. To keep this notebook runnable with no network, we synthesize a small
tidy frame with NumPy.

In [3]:
rng = np.random.default_rng(0)

# Three product lines measured over 12 months: a tidy/long-form frame.
months = pd.date_range("2025-01-01", periods=12, freq="MS")
lines = {"Widgets": 100, "Gadgets": 60, "Gizmos": 40}
rows = []
for name, base in lines.items():
    trend = base + np.arange(12) * rng.uniform(2, 6)      # gentle growth
    noise = rng.normal(0, base * 0.08, 12)
    sales = np.clip(trend + noise, 0, None)
    for m, s in zip(months, sales):
        rows.append({"month": m, "line": name, "sales": round(float(s), 1),
                     "region": rng.choice(["North", "South"])})
sales_df = pd.DataFrame(rows)
print(sales_df.shape)
sales_df.head()

(36, 4)


,month,line,sales,region
0,2025-01-01,Widgets,98.9,South
1,2025-02-01,Widgets,109.7,North
2,2025-03-01,Widgets,109.9,South
3,2025-04-01,Widgets,109.4,South
4,2025-05-01,Widgets,121.1,South


## 5. Worked Examples

### Example 1 — Plotly Express: one call, a full interactive figure

`px.line` maps `month → x`, `sales → y`, and `line → color` straight from column names, returning
a fully-styled, hoverable, legend-toggleable `go.Figure`. We then keep customizing it with
`update_layout` — proving that an Express figure is just a Graph Objects figure underneath.

In [4]:
fig = px.line(
    sales_df, x="month", y="sales", color="line", markers=True,
    title="Monthly sales by product line",
    labels={"sales": "Units sold", "month": "Month"},
)
fig.update_layout(legend_title_text="Product line", hovermode="x unified")

# A px figure IS a go.Figure: inspect the JSON spec it built.
print("type     :", type(fig).__name__)
print("n traces :", len(fig.data))
print("trace[0] :", fig.data[0].type, "->", fig.data[0].name)
fig  # in JupyterLab this renders the interactive chart

type     : Figure
n traces : 3
trace[0] : scatter -> Widgets


### Example 2 — Graph Objects: compose traces by hand

When Express doesn't expose what you need (here: a bar trace and a line trace on the *same* axes,
a dual encoding), drop to `go`. You build each trace explicitly and assemble them into a figure.

In [5]:
totals = sales_df.groupby("line", sort=False)["sales"].sum()
avg = sales_df.groupby("line", sort=False)["sales"].mean()

fig2 = go.Figure()
fig2.add_trace(go.Bar(x=totals.index, y=totals.values, name="Total units",
                      marker_color="#636EFA"))
fig2.add_trace(go.Scatter(x=avg.index, y=avg.values, name="Monthly avg",
                          mode="lines+markers", yaxis="y2",
                          line=dict(color="#EF553B", width=3)))
fig2.update_layout(
    title="Totals (bars) vs monthly average (line) per product line",
    yaxis=dict(title="Total units"),
    yaxis2=dict(title="Avg units/month", overlaying="y", side="right"),
    legend=dict(orientation="h", y=1.12),
)

print("traces:", [t.type for t in fig2.data])
fig2

traces: ['bar', 'scatter']


### Example 3 — faceting + export (no network, no kaleido needed)

`facet_col` splits one chart into a panel per category. Then we export to a **standalone
interactive HTML** string with `write_html` — the artifact you hand to someone without Python.
Static image export (`write_image`) needs **kaleido**, so we gate it on availability instead of
failing the run.

In [6]:
fig3 = px.bar(
    sales_df, x="month", y="sales", color="line",
    facet_col="line", title="Sales per line (faceted)",
)
fig3.update_layout(showlegend=False)

# Standalone interactive HTML — works with just plotly, no extra deps.
html = fig3.to_html(full_html=True, include_plotlyjs="cdn")
print("standalone HTML chars:", len(html))
print("contains plotly.js CDN tag:", "plotly" in html.lower())

# Static PNG export is optional (needs kaleido); show the call shape, gated.
try:
    import kaleido  # noqa: F401
    png = fig3.to_image(format="png", width=700, height=300)
    print("PNG bytes:", len(png))
except Exception as e:
    print("static export skipped (install 'kaleido' to enable):", type(e).__name__)

# Gated network example: the real px.data.gapminder() downloads on first call.
if os.getenv("PLOTLY_ALLOW_NETWORK"):
    gap = px.data.gapminder()
    print("gapminder rows:", len(gap))
else:
    print("set PLOTLY_ALLOW_NETWORK=1 to fetch px.data.gapminder() (skipped offline)")

standalone HTML chars: 10984
contains plotly.js CDN tag: True
static export skipped (install 'kaleido' to enable): ModuleNotFoundError
set PLOTLY_ALLOW_NETWORK=1 to fetch px.data.gapminder() (skipped offline)


## 6. Gotchas & Pitfalls

- **`fig.show()` shows nothing / opens a blank tab.** Rendering depends on the active **renderer**.
  In a non-Jupyter script use `pio.renderers.default = "browser"`, or just
  `fig.write_html("out.html")`. In old JupyterLab you needed an extension; modern (≥3) is fine.
- **`write_image` errors with "kaleido not found."** Static export (PNG/SVG/PDF) is a *separate*
  dependency: `pip install kaleido`. HTML export has no such requirement.
- **Express wants tidy/long data.** `px.line(df, x=..., y=..., color=...)` expects one row per
  observation. A wide frame (a column per series) needs `df.melt()` first, or pass `y=[cols]`.
- **Datetime axes:** pass real `datetime64`/`Timestamp` values, not strings, or Plotly treats the
  axis as categorical and won't space points by time.
- **Huge scatter plots are slow.** `plotly.js` is SVG by default; tens of thousands of points lag.
  Use `go.Scattergl` / `px.scatter(..., render_mode="webgl")` for large point clouds.
- **Mutate, don't rebuild.** After `px.*`, keep tweaking with `fig.update_layout` /
  `update_traces` / `add_trace` — don't reconstruct from scratch.
- **Notebook file size.** Saving a notebook with many embedded interactive figures (each carries
  its data + a copy of plotly.js unless using CDN) bloats the `.ipynb`. Use
  `include_plotlyjs="cdn"` on export, and clear outputs before committing if size matters.
- **`px.data.*` may hit the network** on first use; cache or vendor the data for offline/CI runs.

## 7. When to Use vs Alternatives

| Need | Reach for | Why |
|------|-----------|-----|
| Interactive exploration (zoom/hover/toggle) | **Plotly** | Built-in, zero JS to write |
| Print/paper-ready static figure | **Matplotlib** | Lighter, full vector control, no JS payload |
| Quick statistical chart from a DataFrame | **seaborn** | Terser for stats (CIs, regressions, KDE) |
| Full web dashboard / app | **Dash** (Plotly) or **Streamlit** | Dash is Plotly-native; reactive layout |
| Grammar-of-graphics, declarative-only | **Altair** (Vega-Lite) | Cleaner grammar; less imperative control |
| Browser interactivity inside the Python notebook ecosystem | **Bokeh** | Comparable; different API/feel |
| Millions of points | **datashader** / **Scattergl** | Rasterize/aggregate before drawing |

**Honest trade-offs.** Plotly's superpower is *interactivity for free* and a single object model
shared with Dash. Costs: heavier output (an interactive figure embeds data + JS), a static-export
dependency (kaleido), and less polished defaults than seaborn for pure statistical summaries.
Altair offers a cleaner declarative grammar but less escape-hatch control; Matplotlib wins for
static, publication-grade output. For *exploration and sharing interactive results*, Plotly is the
default; for *a figure in a PDF*, it usually isn't.

## 8. Resources

- **Official Python docs** — https://plotly.com/python/  (every chart type with runnable code)
- **Plotly Express API reference** — https://plotly.com/python-api-reference/plotly.express.html
- **Graph Objects / Figure reference** — https://plotly.com/python/graph-objects/
- **Figure structure & the JSON spec** — https://plotly.com/python/figure-structure/
- **Dash (build dashboards from these figures)** — https://dash.plotly.com/
- **Renderers & static image export (kaleido)** — https://plotly.com/python/renderers/ and
  https://plotly.com/python/static-image-export/